# 第03章 图像运算（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 3 章的跟随代码。
> **全量收录原书代码**；原 notebook 缺失的例题（例 3.8、3.12、3.14、3.16）与按位非/异或、整图加常数、阈值处理等独立演示已按书补全，内容依据站点笔记 ch03.md。
> `cv2.imshow` 需在**本地 Python 环境**运行（弹窗显示图像，网页预览无法显示）；**工作目录须为「数字图像处理-python」根目录**（picture1.jpg、picture3.jpg、watermark.jpg 等素材都在这里），否则 imread 返回 None。

In [ ]:
# ===== 通用显示函数（本地运行显示窗口用）=====
import cv2

def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

## 3.1 图像加法运算

两幅图相加，对应像素的和很容易超过 255——溢出怎么办，两种写法给了两个相反的答案：

- `a + b`（加号运算符）：对 256 **取模**（Modulo Arithmetic），亮的像素会回绕变暗；
- `cv2.add(a, b)`：**饱和运算**（Saturating Arithmetic），超过 255 截断为 255。

取模的根源是 dtype：`np.uint8` 只有 8 个比特位，第 9 位（权值 256）无处安放直接被丢掉。本节含例 3.1–3.3。

### 例 3.1 · 加号运算符：对 256 取模

用随机数组模拟灰度图（`dtype=np.uint8`）：`img1 + img2` 逐元素相加，超过 255 的元素对 256 取模——结果可能比两个加数都小，这就是"两幅图相加却变暗"的机制。

In [ ]:
# ===== 3.1.1 例 3.1：加号运算符，溢出对 256 取模 =====
import numpy as np

# 3×3 的"假灰度图"，dtype 必须写 uint8，才能复现 8 位图像的行为
img1 = np.random.randint(0, 256, size=[3, 3], dtype=np.uint8)
img2 = np.random.randint(0, 256, size=[3, 3], dtype=np.uint8)
print("img1=\n", img1)
print("img2=\n", img2)
print("img1+img2=\n", img1 + img2)      # 超过 255 的元素被对 256 取模

### 例 3.2 · cv2.add()：饱和运算

`dst = cv2.add(src1, src2[, dst[, mask[, dtype]]])`——溢出截断到饱和值 255，只会"到顶"、不会"掉下来"。同一组输入，与例 3.1 的取模结果对比一目了然。

In [ ]:
# ===== 3.1.2 例 3.2：cv2.add 饱和加法 =====
import numpy as np
import cv2

img1 = np.random.randint(0, 256, size=[3, 3], dtype=np.uint8)
img2 = np.random.randint(0, 256, size=[3, 3], dtype=np.uint8)
print("img1=\n", img1)
print("img2=\n", img2)
print("cv2.add(img1,img2)=\n", cv2.add(img1, img2))   # 超过 255 的统统压到 255

### 例 3.3 · 两种加法用在真实图像上

拿一幅图与它自己相加：`a + b` 取模整体变暗（大片暗斑噪点），`cv2.add` 饱和整体变亮（高光细节丢失）。想让图变亮就用 `cv2.add()`。注意 `b = a` 只是别名，两者是同一块内存。

In [ ]:
# ===== 3.1.2 例 3.3：真实图像上的取模 vs 饱和 =====
import cv2

# 工作目录需为 数字图像处理-python 根目录
a = cv2.imread("picture1.jpg", 0)     # 0：读成单通道灰度图
b = a                                 # 注意：这是别名，b 和 a 是同一块内存
result1 = a + b                       # 取模：超过 255 的部分回绕
result2 = cv2.add(a, b)               # 饱和：超过 255 的压到 255

print("原图 mean:", round(a.mean(), 2))       # 186.91
print("a+b  mean:", round(result1.mean(), 2)) # 169.92：整体变暗
print("add  mean:", round(result2.mean(), 2)) # 236.16：整体变亮

cv2.imshow("original", a)
cv2.imshow("result1", result1)
cv2.imshow("result2", result2)
cv2.waitKey()
cv2.destroyAllWindows()

## 3.2 图像加权和

图像加权和（Weighted Sum）也叫图像混合（Image Blending）/ 图像融合：

```text
dst = cv2.addWeighted( src1, alpha, src2, beta, gamma[, dst[, dtype]] )
```

即 `dst = saturate(src1×α + src2×β + γ)`。两幅图**必须大小、类型相同**；`gamma` 是亮度调节量，**可以是 0 但必须写，不能省略**；`alpha + beta` 可以不等于 1——和为 1 亮度不变，小于 1 变暗，大于 1 变亮直至饱和。本节含例 3.4–3.6。

### 例 3.4 · 用常数数组验证算式

`img1` 全 100、`img2` 全 10，权重 0.6 与 5、`gamma=3`：100×0.6 + 10×5 + 3 = 113。注意权重和 5.6 远大于 1——本例基数小才没顶到上限，换成亮图同样的系数会大片压成 255。

In [ ]:
# ===== 3.2 例 3.4：常数数组验证 addWeighted 算式 =====
import cv2
import numpy as np

img1 = np.ones((3, 4), dtype=np.uint8) * 100   # 3×4、值全为 100 的"灰度图"
img2 = np.ones((3, 4), dtype=np.uint8) * 10    # 3×4、值全为 10
gamma = 3                                      # 亮度调节量

img3 = cv2.addWeighted(img1, 0.6, img2, 5, gamma)
print(img3)

### 例 3.5 · 两幅真实图像按 0.6 / 0.4 混合

`addWeighted` 要求尺寸一致，先把第二幅 `cv2.resize` 到第一幅的大小。0.6 + 0.4 = 1，整体亮度基本不变；把 alpha 从 0 连续调到 1、beta 取 1−alpha，就得到从 a 渐变到 b 的动画。

In [ ]:
# ===== 3.2 例 3.5：两幅真实图像加权混合 =====
import cv2

a = cv2.imread("picture1.jpg")                                  # 1920×1080
b = cv2.resize(cv2.imread("picture3.jpg"), (a.shape[1], a.shape[0]))  # 缩放到同尺寸

result = cv2.addWeighted(a, 0.6, b, 0.4, 0)   # 0.6+0.4=1，整体亮度基本不变
print(result.shape, result.dtype)             # (1080, 1920, 3) uint8

cv2.imshow("a", a)
cv2.imshow("b", b)
cv2.imshow("result", result)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 3.6 · 只混合局部：ROI 加权混合

先切片取出两块同尺寸的 ROI（Region of Interest，感兴趣区域），`addWeighted` 混合后写回。切片顺序是 `[行, 列]` 即 `[y, x]`：`150:550` 是高度方向（行），写反了会取到完全不相干的一块。

In [ ]:
# ===== 3.2 例 3.6：只混合局部（ROI 加权混合）=====
import cv2

lena = cv2.imread("picture3.jpg")     # 人像素材
bg = cv2.imread("picture1.jpg")       # 另一幅图

face1 = lena[150:550, 200:600]        # 人像的脸部区域，400×400
face2 = bg[250:650, 850:1250]         # 另一幅图里同尺寸的区域，400×400

add = cv2.addWeighted(face1, 0.6, face2, 0.4, 0)   # 两块 ROI 加权混合
lena2 = lena.copy()                                # 先复制，避免改到原图
lena2[150:550, 200:600] = add                      # 把混合结果写回

cv2.imshow("lena", lena)
cv2.imshow("result", lena2)
cv2.waitKey()
cv2.destroyAllWindows()

## 3.3 按位逻辑运算

按位运算（Bitwise Operation）把数值转成二进制后，在对应比特位上做逻辑运算——取某一位、保留某块区域、写水印、加密全靠它。四个函数：

- `cv2.bitwise_and()`：按位与——掩模，非零区域保留、其余置零；
- `cv2.bitwise_or()`：按位或——合并、把信息写进指定的位；
- `cv2.bitwise_xor()`：按位异或——加密解密、找差异（自反：两次异或还原）；
- `cv2.bitwise_not()`：按位非——反色（底片效果），等价 255−N。

按位与的两条铁律：任何数 N 与 0 相与得 0；与 255（1111 1111）相与得 N 自身。本节含例 3.7–3.9 与两个小演示。

### 例 3.7 · 用数组验证掩模效果

掩模 b 只有 0 和 255：`bitwise_and(a, b)` 后，b 为 255 的位置原样保留 a 的值，b 为 0 的位置全变 0——这就是"掩模"的本质。

In [ ]:
# ===== 3.3.1 例 3.7：掩模效果验证 =====
import cv2
import numpy as np

a = np.random.randint(0, 255, (5, 5), dtype=np.uint8)   # 模拟灰度图
b = np.zeros((5, 5), dtype=np.uint8)                     # 掩模：先全 0
b[0:3, 0:3] = 255                                        # 左上 3×3 打开
b[4, 4] = 255                                            # 右下角单独打开

c = cv2.bitwise_and(a, b)
print("a=\n", a)
print("b=\n", b)
print("c=\n", c)

### 例 3.8 · 给真实灰度图做掩模

与 a 同尺寸的全零掩模，两块矩形区域置 255；按位与后掩模内保留原图、掩模外全黑。

In [ ]:
# ===== 3.3.1 例 3.8：真实灰度图的掩模提取 =====
import cv2
import numpy as np

a = cv2.imread("picture1.jpg", 0)              # 灰度图（1080×1920）
b = np.zeros(a.shape, dtype=np.uint8)          # 与 a 同尺寸的掩模
b[150:700, 800:1300] = 255                     # 掩模区域一
b[400:900, 600:800] = 255                      # 掩模区域二

c = cv2.bitwise_and(a, b)                      # 掩模内保留，掩模外全黑
cv2.imshow("a", a)
cv2.imshow("b", b)
cv2.imshow("c", c)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 3.9 · 彩色图像：掩模扩成 3 通道

按位运算要求两个操作数**通道数相同**，3 通道 BGR 图不能直接与单通道掩模按位与（报 `(-209:Sizes of input arguments do not match)`）——把掩模直接建成 3 通道即可。更省事的写法（单通道掩模传 mask 参数）见 3.4 节例 3.11。

In [ ]:
# ===== 3.3.1 例 3.9：掩模扩成 3 通道后按位与 =====
import cv2
import numpy as np

a = cv2.imread("picture1.jpg", 1)              # 3 通道 BGR
b = np.zeros(a.shape, dtype=np.uint8)          # 掩模直接建成 3 通道
b[150:700, 800:1300] = 255
b[400:900, 600:800] = 255

c = cv2.bitwise_and(a, b)                      # 形状一致，可以按位与
print("a.shape=", a.shape)                     # (1080, 1920, 3)
print("b.shape=", b.shape)                     # (1080, 1920, 3)

### 按位非：反色（底片）效果

`dst = cv2.bitwise_not(src[, dst[, mask]])`——逐位取反，对 8 位无符号整数等价于 `255 − N`（198 → 57），即常见的反色效果。按位操作只对整型有意义，传浮点数组结果毫无意义。

In [ ]:
# ===== 3.3.3 按位非：反色（底片）效果 =====
import cv2

img = cv2.imread("picture1.jpg")
inv = cv2.bitwise_not(img)          # 三个通道分别取反
print(round(img.mean(), 2))         # 187.39
print(round(inv.mean(), 2))         # 67.61  正好是 255-187.39

### 按位异或：自反性验证

异或（XOR）也叫半加运算（Half Addition）：两位**不同**结果 1、**相同**结果 0。自反性 `xor(xor(a,b),b)=a` 是它成为"加密工具"的根本——3.7 节的加密与 3.9 节的打码解码都建立在这条性质上。

In [ ]:
# ===== 3.3.4 按位异或：自反性验证 =====
import cv2
import numpy as np

a = np.array([[198]], dtype=np.uint8)
b = np.array([[219]], dtype=np.uint8)
print((a ^ b)[0, 0])                     # 29
print(cv2.bitwise_xor(a, b)[0, 0])       # 29，与 numpy 的 ^ 等价
print(cv2.bitwise_xor(cv2.bitwise_xor(a, b), b)[0, 0])   # 198：两次异或还原

## 3.4 掩模

OpenCV 里很多函数都带 **mask** 参数（掩模，也称掩码）。规则一句话：**只处理掩模值非零的像素，其余像素的输出置 0**；掩模值 1 与 255 等价，但要把掩模当图像显示就得乘 255。

彩色图配单通道掩模有两种写法：把掩模扩成 3 通道逐通道按位与（例 3.9），或用 `mask` 参数——`cv2.bitwise_and(a, a, mask=mask)`（例 3.11，原理是任何数值与自身按位与仍是自身）。注意 `mask` 必须**关键字传参**：bitwise_and 的第三个位置参数是 dst。

本节含例 3.10–3.11。

### 例 3.10 · 掩模参数演示

img1 全 3、img2 全 5，mask 只有右下 2×2 非零：`cv2.add(img1, img2, mask=mask)` 只在掩模内算 3+5=8，其余一律 0。先给 img3 赋初值 66 是为了说明"输出是新数组"——初值不参与运算、直接被丢弃。

In [ ]:
# ===== 3.4 例 3.10：掩模参数演示 =====
import cv2
import numpy as np

img1 = np.ones((4, 4), dtype=np.uint8) * 3
img2 = np.ones((4, 4), dtype=np.uint8) * 5

mask = np.zeros((4, 4), dtype=np.uint8)
mask[2:4, 2:4] = 1                     # 只有右下 2×2 为非零

img3 = np.ones((4, 4), dtype=np.uint8) * 66   # 初值只为说明"会被丢弃"
print("初始值 img3=\n", img3)

img3 = cv2.add(img1, img2, mask=mask)
print("求和后 img3=\n", img3)

### 例 3.11 · 用掩模参数处理彩色图

单通道掩模直接传给 `mask` 参数，作用到 3 通道图像时对三个通道一视同仁；掩模的**行、列数必须与图像一致**，否则报错。`w, h, ch = a.shape` 沿用书中命名：w 其实是行数（高）。

In [ ]:
# ===== 3.4 例 3.11：掩模参数处理彩色图 =====
import cv2
import numpy as np

a = cv2.imread("picture1.jpg", 1)          # 3 通道
w, h, ch = a.shape                         # 书中沿用 w/h 命名：w 其实是行数（高）

mask = np.zeros((w, h), dtype=np.uint8)    # 单通道掩模
mask[150:700, 800:1300] = 255
mask[400:900, 600:800] = 255

c = cv2.bitwise_and(a, a, mask=mask)       # mask 必须关键字传参（第三个位置参数是 dst）
print("a.shape=", a.shape)                 # (1080, 1920, 3)
print("mask.shape=", mask.shape)           # (1080, 1920)

cv2.imshow("a", a)
cv2.imshow("mask", mask)
cv2.imshow("c", c)
cv2.waitKey()
cv2.destroyAllWindows()

## 3.5 图像与数值的运算

加法运算与按位运算里，两个操作数可以都是图像，也可以是一图配一数（顺序无关）：`cv2.add(img1, img2)`、`cv2.add(img1, 6)`、`cv2.add(6, img2)`。数值同样服从饱和规则；对称的 `cv2.subtract()` 也是饱和的（5−10=0，不会回绕成 251）。

### 例 3.12 · 三种形式跑一遍 + 整图加 100 的对比

前一个 cell 把三种参数组合各跑一遍；后一个 cell 给整幅灰度图加 100：`cv2.add` 饱和提亮，`img + 100` 模运算回绕反而变暗。

原书笔误提醒：书上把加固定值演示成 `cv2.add(333, img)`——cv2 里能跑（标量按饱和处理，全图变 255），但 numpy 的 `333 + img` 在 NumPy 2.x 直接抛 OverflowError；想加超过 255 的值，先把图转 int16。

In [ ]:
# ===== 3.5 例 3.12：add 的三种参数形式 =====
import cv2
import numpy as np

img1 = np.ones((4, 4), dtype=np.uint8) * 3
img2 = np.ones((4, 4), dtype=np.uint8) * 5

print("cv2.add(img1,img2)=\n", cv2.add(img1, img2))   # 全 8
print("cv2.add(img1,6)=\n", cv2.add(img1, 6))         # 全 9
print("cv2.add(6,img2)=\n", cv2.add(6, img2))         # 全 11

In [ ]:
# ===== 3.5 例 3.12（续）：整图加 100，饱和 vs 取模 =====
import cv2

img = cv2.imread("picture1.jpg", 0)

bright_add = cv2.add(img, 100)   # 饱和：超过 255 的截到 255
bright_np = img + 100            # 模运算：超过 255 的回绕到小值

print(round(img.mean(), 2))         # 186.91
print(round(bright_add.mean(), 2))  # 239.15  提亮
print(round(bright_np.mean(), 2))   # 99.49   反而变暗

## 3.6 位平面分解

把灰度图所有像素的**同一个比特位**抽出来重新组成一幅二值图，就是一个位平面（Bit Plane）；8 位图共 8 层。第 7 位（最高位，权重 128）与原图相关性最高、最像原图；第 0 位（权重 1）通常杂乱无章、近似噪声。

分解五步：读图取宽高 → 构造提取矩阵（值为 2 的 n 次幂）→ 按位与只留第 n 位 → 阈值处理（大于零改 255）→ 显示/保存。

先看一个 2×2 小例子体会阈值处理：提取出来的值只可能是 0 或 2 的幂，直接显示全是近黑，必须阈值到 255 才看得见。

In [ ]:
# ===== 3.6 阈值处理：非零统一改成 255 才看得见 =====
import numpy as np

plane = np.array([[0, 8], [8, 0]], dtype=np.uint8)   # 已提取的位平面：值只可能是 0 或 8
mask = plane > 0        # 大于 0 → True，否则 False
plane[mask] = 255       # 布尔掩模索引，把 True 位置改成 255
print(plane)            # [[  0 255] [255   0]]

### 例 3.13 · 一次取出 8 个位平面

提取矩阵第 i 个通道的值全是 2 的 i 次幂，逐层按位与后立刻阈值到 255。注意阈值必须写成 `planes[:, :, i][mask] = 255` 只作用当前层——直接写 `planes[mask] = 255` 会把 8 个层一起污染（原书正文 `RD[:,:,i] > 0` 是笔误，二维位平面用 `RD > 0` 即可）。

In [ ]:
# ===== 3.6 例 3.13：位平面分解，一次取出 8 个 =====
import cv2
import numpy as np

lena = cv2.imread("picture1.jpg", 0)      # 灰度读入
r, c = lena.shape

# 提取矩阵：r×c×8，第 i 个通道的值全是 2 的 i 次幂
x = np.zeros((r, c, 8), dtype=np.uint8)
for i in range(8):
    x[:, :, i] = 2 ** i

planes = np.zeros((r, c, 8), dtype=np.uint8)
for i in range(8):
    planes[:, :, i] = cv2.bitwise_and(lena, x[:, :, i])   # 只留第 i 位
    mask = planes[:, :, i] > 0                            # 非零即"该位为 1"
    planes[:, :, i][mask] = 255                           # 阈值到 255，只作用当前层
    cv2.imshow(str(i), planes[:, :, i])                   # 窗口名就用位平面序号
cv2.waitKey()
cv2.destroyAllWindows()

## 3.7 图像加密和解密

异或的自反性直接拿来做加密：**加密** = 明文与密钥按位异或得密文；**解密** = 密文与同一把密钥再异或一次得明文。密钥图必须与原图同尺寸、要随机（密文均值趋于 127.5、直方图趋于均匀）；若用纯色图当密钥，密文的明暗结构会原样保留。

### 例 3.14 · 用随机数生成密钥图像，加密后再解密

In [ ]:
# ===== 3.7 例 3.14：图像加密和解密 =====
import cv2
import numpy as np

lena = cv2.imread("picture1.jpg", 0)      # 明文
r, c = lena.shape
key = np.random.randint(0, 256, size=[r, c], dtype=np.uint8)   # 密钥图，与原图同尺寸

encryption = cv2.bitwise_xor(lena, key)          # 加密
decryption = cv2.bitwise_xor(encryption, key)    # 用同一密钥解密

print(np.array_equal(decryption, lena))          # True：无损还原
print(round(encryption.mean(), 1))               # 127.4：密文均值趋于 127.5
print(round(lena.mean(), 1))                     # 186.9

cv2.imshow("lena", lena)
cv2.imshow("key", key)
cv2.imshow("encryption", encryption)
cv2.imshow("decryption", decryption)
cv2.waitKey()
cv2.destroyAllWindows()

## 3.8 数字水印

**最低有效位（LSB）信息隐藏**：把二值水印写进载体图像的最低有效位——即用二值图替换载体的第 0 个位平面。每个像素最多变动 1，人眼察觉不到（不可见水印），用于信息隐藏、版权认证。

- **嵌入**：载体与 254（1111 1110）按位与，高七位保留、最低位清零；水印二值化（255→1）后与载体按位**或**，写入最低位；
- **提取**：含水印图与 1（0000 0001）按位与，只剩最低位；再把 1 阈值成 255 显示。

LSB 水印鲁棒性差：存成 JPEG 提取错误率近 30%，请存 PNG/BMP。本节含例 3.15。

### 例 3.15 · 完整模拟水印的嵌入与提取

载体图与 `watermark.jpg` 都是 1920×1080，正好配套。提取出的水印与原始二值水印逐像素一致；含水印图与原图最大差值 1、PSNR 约 51 dB，肉眼看不出区别。

In [ ]:
# ===== 3.8 例 3.15：LSB 水印嵌入与提取 =====
import cv2
import numpy as np

# 载体图像（灰度）与水印图像（白底黑字）
lena = cv2.imread("picture1.jpg", 0)
watermark = cv2.imread("watermark.jpg", 0)

# ① 水印二值化：255 → 1，方便写进 1 个比特位
w = watermark[:, :] > 0
watermark[w] = 1

r, c = lena.shape

# ========== 嵌入 ==========
t254 = np.ones((r, c), dtype=np.uint8) * 254   # 1111 1110
lenaH7 = cv2.bitwise_and(lena, t254)           # 高七位保留，最低位清零
e = cv2.bitwise_or(lenaH7, watermark)          # 用或运算把水印写进最低位

# ========== 提取 ==========
t1 = np.ones((r, c), dtype=np.uint8)           # 0000 0001
wm = cv2.bitwise_and(e, t1)                    # 只留最低位：得到 0/1 图
print(wm.max(), wm.min())                      # 1 0

w = wm[:, :] > 0                               # 1 → 255，方便显示
wm[w] = 255

cv2.imshow("lena", lena)
cv2.imshow("watermark", watermark * 255)       # watermark 现在最大是 1，乘 255 才看得见
cv2.imshow("e", e)
cv2.imshow("wm", wm)
cv2.waitKey()
cv2.destroyAllWindows()

## 3.9 脸部打码及解码

用**掩模 + 按位运算**收口的综合例子，思路四步：整幅图与密钥异或得全密文 → 掩模抠出脸部密文 → 反掩模抠出非脸部原图 → 两块相加得"只有脸是密文"的打码图。解码是其逆过程：打码图整幅再与密钥异或，掩模抠脸部明文、反掩模抠非脸部，两块相加还原。

拼两块区域书里用加号，本例因两块互斥结果相同；真遇到重叠区域，加号会在 uint8 下回绕出错，用按位或更稳妥。

### 例 3.16 · 脸部打码与解码

mask 脸部区域先存 1、显示时乘 255；解码结果与原图完全一致（`array_equal` 为 True）。12 个中间变量（lena / mask / key / lenaXorKey / encryptFace / noFace1 / maskFace / extractOriginal / extractFace / noFace2 / extractLena 等）的含义见站点笔记的对照表。

In [ ]:
# ===== 3.9 例 3.16：脸部打码及解码 =====
import cv2
import numpy as np

lena = cv2.imread("picture3.jpg", 0)      # 人像素材，灰度
r, c = lena.shape

mask = np.zeros((r, c), dtype=np.uint8)
mask[250:750, 120:620] = 1                # 脸部区域：先存 1，显示时乘 255

key = np.random.randint(0, 256, size=[r, c], dtype=np.uint8)   # 与图同尺寸的密钥

# ========== 打码 ==========
lenaXorKey = cv2.bitwise_xor(lena, key)                  # 整幅图异或成密文
encryptFace = cv2.bitwise_and(lenaXorKey, mask * 255)     # 只留脸部密文
noFace1 = cv2.bitwise_and(lena, (1 - mask) * 255)         # 只留非脸部原图
maskFace = encryptFace + noFace1                          # 拼成打码图

# ========== 解码 ==========
extractOriginal = cv2.bitwise_xor(maskFace, key)          # 整幅图再异或一次
extractFace = cv2.bitwise_and(extractOriginal, mask * 255)  # 抠出脸部明文
noFace2 = cv2.bitwise_and(maskFace, (1 - mask) * 255)       # 抠出非脸部
extractLena = noFace2 + extractFace                        # 拼回原图

print(np.array_equal(extractLena, lena))                   # True：完全还原